In [ ]:
import cv2
import matplotlib.pyplot as plt
import ultralytics
import numpy as np

ultralytics.checks()

In [ ]:
# Загрузка моделей для детекции поля и игроков
field_detection_model = ultralytics.YOLO('../../weights/field_detection.pt')
player_detection_model = ultralytics.YOLO('../../weights/player_detection.pt')

# Предсказание местоположения поля на видео
field_detection_results = field_detection_model.predict(
    source='../../акбарс_цска_короткое_2.mp4',
    conf=0.3,
    iou=0.5,
    device='mps',
    save=True,
)

# Предсказание местоположения игроков на видео
player_detection_results = player_detection_model.predict(
    source='../../акбарс_цска_короткое_2.mp4',
    conf=0.3,
    device='mps',
    # save=True,
)

In [ ]:
# # Словарь с координатами ключевых точек на 2D поле

# real_field_coordinates = {
#     'left_vorota': [0.08, 0.5],
#     'left_up_red_circle': [0.176, 0.276],
#     'left_down_red_circle': [0.176, 0.724],
#     'left_down_red_circle_up_point': [0.24, 0.650],

#     'center': [0.5, 0.5],
    
#     'right_vorota': [0.920, 0.5],
#     'right_up_red_circle': [0.824, 0.276],
#     'right_down_red_circle': [0.824, 0.724],
#     'right_down_red_circle_up_point': [0.76, 0.650],

# }

In [ ]:
# real_field_coordinates = {
#     'left_goal': [0.08, 0.5],
#     'left_top_red_circle': [0.176, 0.276],
#     'left_bottom_red_circle': [0.176, 0.724],
#     'left_bottom_red_circle_upper_point': [0.24, 0.650],

#     'center_left_top_red_circle': [0.4, 0.4],
#     'center_left_bottom_red_circle': [0.4, 0.6],

#     'center': [0.5, 0.5],
    
#     'right_goal': [0.920, 0.5],
#     'right_top_red_circle': [0.824, 0.276],
#     'right_bottom_red_circle': [0.824, 0.724],
#     'right_bottom_red_circle_upper_point': [0.76, 0.650],
# }


real_field_coordinates = {
    'left_goal': [0.06, 0.5],
    'left_top_red_circle': [0.17, 0.246],
    'left_bottom_red_circle': [0.17, 0.754],
    'left_bottom_red_circle_upper_point': [0.234, 0.66],

    'center_left_top_red_circle': [0.375, 0.25],
    'center_left_bottom_red_circle': [0.375, 0.75],

    'center_left_top_red_circle': [0.375, 0.25],
    'center_left_bottom_red_circle': [0.375, 0.75],

    'center': [0.5, 0.5],
    
    'right_goal': [0.920, 0.5],
    'right_top_red_circle': [0.824, 0.276],
    'right_bottom_red_circle': [0.824, 0.724],
    'right_bottom_red_circle_upper_point': [0.76, 0.650],
}

In [ ]:
field_image = cv2.imread('field_3.png')
field_image = cv2.resize(field_image, (400, int(400 / (675 / 348))))
field_image_height, field_image_width, _ = field_image.shape

In [ ]:
# Открытие видео и подготовка для записи

video_path = '../../акбарс_цска_короткое_2.mp4'
video_capture = cv2.VideoCapture(video_path)
video_codec = cv2.VideoWriter_fourcc(*'mp4v')
output_video = cv2.VideoWriter('output_video.mp4', video_codec, 30.0, 
                               (int(video_capture.get(3)), int(video_capture.get(4))))


# Обработка каждого кадра
for frame_index in range(len(field_detection_results)):
    # Получаем текущий кадр
    current_frame = field_detection_results[frame_index].orig_img

    
    if frame_index % 5 == 0:
        # Отображаем текущий кадр
        plt.imshow(cv2.cvtColor(current_frame, cv2.COLOR_BGR2RGB))
        plt.show()

    # Накладываем изображение поля на текущий кадр
    current_frame[:field_image_height, :field_image_width] = field_image
    frame_height, frame_width, _ = current_frame.shape

    # Получаем текущие результаты детекции
    current_field_result = field_detection_results[frame_index]
    current_player_result = player_detection_results[frame_index]

    # Извлечение классов объектов на поле
    field_classes = current_field_result.boxes.cls.to('cpu').numpy()
    
    # Идентификация кругов и ворот на поле
    circle_indices = (field_classes == 5)
    goal_indices = (field_classes == 6)

    # Координаты игроков на поле
    player_coordinates = current_player_result.boxes.xyxyn.to('cpu').numpy()
    players_x_center = player_coordinates[:, (0, 2)].sum(axis=1) / 2
    players_y_bottom = player_coordinates[:, 3]
    player_coordinates = np.column_stack((players_x_center, players_y_bottom))

    # Проверка наличия нужных объектов (двух кругов и одних ворот)
    if sum(circle_indices) >= 2 and sum(goal_indices) == 1:
        # Извлечение координат объектов
        circle_coordinates = current_field_result.boxes.xywhn[circle_indices].to('cpu').numpy()
        goal_coordinates = current_field_result.boxes.xywhn[goal_indices].to('cpu').numpy()[0]
        goal_x = goal_coordinates[0]

        # Средняя координата по x для кругов
        circle_average_x = circle_coordinates[:, 0].mean()
    
        # Сортировка координат кругов
        circle_coordinates = circle_coordinates[:, :2].tolist()

        # print(f'CIRCLE: {circle_coordinates}')
        
        # Определение половины поля, на которой идет игра
        if circle_average_x > goal_x:
            circle_coordinates.sort(key=lambda x: x[0])
            # print('Игра в левой половине')
            field_side = 'left'
        else:
            circle_coordinates.sort(key=lambda x: x[0], reverse=True)
            # print('Игра в правой половине')
            field_side = 'right'

        # Определение верхнего и нижнего круга
        top_circle, bottom_circle = sorted(circle_coordinates[:2], key=lambda c: c[1])


        # Проверить что 2 задетекченных круга действительно находятся близко к воротам:
        if (top_circle[0] > goal_coordinates[0] + goal_coordinates[2] * 13 or bottom_circle[0] > goal_coordinates[0] + goal_coordinates[2] * 13) and field_side == 'left':
            cv2.circle(current_frame, (int((goal_coordinates[0] + goal_coordinates[2] * 13)*frame_width), int(goal_coordinates[1] * frame_height)), radius=5, color=(255,0,0),thickness=-1)
            output_video.write(current_frame)
            continue

        # Координаты ворот
        goal_xy = goal_coordinates[:2]
        goal_xy[1] += goal_coordinates[3] * 0.2
        goal_xy[0] += goal_coordinates[2] * 0.1

        # Определение координат для Affine Transformation
        detected_image_points = [top_circle, bottom_circle, goal_xy]

        # Реальные координаты на 2D поле
        real_world_points = [
            real_field_coordinates['left_top_red_circle'],
            real_field_coordinates['left_bottom_red_circle'],
            real_field_coordinates['left_goal']
        ]

        other_circles = circle_coordinates[2:]

        if len(other_circles) <= 2:

            for circle in other_circles:

                if abs(circle[1] - top_circle[1]) < abs(circle[1] - bottom_circle[1]):
                    circle_name = 'center_left_top_red_circle'
                else:
                    circle_name = 'center_left_bottom_red_circle'

                detected_image_points.append(circle)
                real_world_points.append(real_field_coordinates[circle_name])

        if len(detected_image_points) == 3:
            # Вычисление аффинного преобразования
            affine_transform_matrix = cv2.getAffineTransform(np.float32(detected_image_points), np.float32(real_world_points))
            transformed_player_coordinates = cv2.transform(np.float32([player_coordinates]), affine_transform_matrix)[0]
        else:
            homography_matrix, _ = cv2.findHomography(np.float32(detected_image_points), np.float32(real_world_points))
            transformed_player_coordinates = cv2.perspectiveTransform(np.float32([player_coordinates]), homography_matrix)[0]


        for player_x, player_y in transformed_player_coordinates:
            current_frame = cv2.circle(current_frame, 
                                    center=(int(player_x * field_image_width), int(player_y * field_image_height)), 
                                    radius=5, color=(0, 0, 255), thickness=-1)

        for point in detected_image_points:
            current_frame = cv2.circle(current_frame, 
                                    center=(int(point[0] * frame_width), int(point[1] * frame_height)), 
                                    radius=5, color=(0, 255, 0), thickness=-1)

    else:
        print('Игра в центре')

    # Сохранение кадра в выходное видео
    output_video.write(current_frame)

# Закрытие выходного видео
output_video.release()